This notebook generates all plots regarding the ABC-SMC-based parameter inference and mutant search, plotting prior/posterior distributions of the parameters of WT IL-2 and its studied mutants. Also contains the distribution of EC50/amplitude among the prior/posterior parameter distributions and the prior/posterior distributions of the mutant search

In [1]:
import pyabc
from sympy import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import use
use('Agg')
import warnings
from src.simulate_func import update_df_bind_ABC
warnings.filterwarnings('ignore')
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14) 

# Plot and save parameter inference of WT alone

In [2]:
directory_plots = "figures/ABC_SMC/"
conf95_2SD = 0.75
df_bind = pd.read_csv("data/binding/IL2_data_param.csv")
param_dict = {}
variants_list = ["WT"]
param_depen = {'k_IL_RB_b':['k_IL_RA_RB_b'], 'k_RA_RB_b':['k_IL_RA_RB_b'], 'k_IL_RB_RG_b':['k_IL_RA_RB_RG_b'], 'k_IL_RA_RB_RG_b':['k_IL_RB_RG_b']}
param_unique_var = {"WT": df_bind.columns[1:],"NKTR-358":['k_IL_RA_b','k_IL_RB_b','k_IL_RA_RB_b'],"WT_Nterm":['k_IL_RA_b','k_IL_RB_RG_b','k_IL_RA_RB_RG_b'],"MDNA11":['k_IL_RA_f','k_IL_RA_b','k_IL_RB_b'],"ALKS4230":['k_IL_RA_f','k_IL_RA_b'],"THOR-707":['k_IL_RA_f','k_IL_RA_b','k_IL_RB_f','k_IL_RB_b','k_IL_RA_RB_b']}

In [3]:
for variant in variants_list:
    # Load parameter inference data
    history = pyabc.History("sqlite:///results/fit_ABC_SMC/results/pyABC_inference__"+variant+".db")
    df_fit_data = pd.read_csv("results/fit_ABC_SMC/IL2_data_fit.csv") # Parameters to fit
    df_fit_data["Var__Param"] = df_fit_data["Variant"]+"__"+df_fit_data["Parameter"]
    df_fit_data = df_fit_data[["Var__Param","Initial_val","Lower_bound","Upper_bound"]]
    df_fit_data = df_fit_data.loc[df_fit_data["Var__Param"].str.contains(variant)]
    
    # For each infered parameter in the variant
    for var__param in df_fit_data["Var__Param"]:
        var, param = var__param.split("__")

        # Plot prior and final posterior distributions
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        
        # Plot prior
        x = np.linspace(np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2,np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2,100)
        pdf = pyabc.RV("norm", loc=np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]), scale=conf95_2SD/2).pdf(x)
        # If infered parameter is an unbinding rate in the plot we put the Kd
        if (param.split('_')[0]=='k') & (param.split('_')[-1]=='b'): 
            x = 10**x/df_bind.loc[df_bind["Variant"]==var,"_".join(param.split('_')[:-1])+"_f"].values[0]
        else:
            x = 10**x
        opt_param_evol = [x[np.argmax(pdf)]] # Save value where prior is maximum for temporal evolution    
            
        # Plot prior
        plt.plot(x,pdf/np.sum(pdf), color="darkred", linewidth=7.5, label=r"$p(\theta)$")
        
        print("Original "+var+"_"+param+": " + str(x[np.argmax(pdf)]))
        
        # Plot posterior
        df, w = history.get_distribution(m=0, t=history.max_t)
        x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=100)
        # If infered parameter is an unbinding rate in the plot we put the Kd
        if (param.split('_')[0]=='k') & (param.split('_')[-1]=='b'):
            # Change the binding rates dataset to include the fitted parameters
            param_dict[var__param] = x[np.argmax(pdf)]
            
            # Get the evolution of the optimal parameter values with each iteration of the posterior
            for i in range(0,history.max_t+1):
                df, w = history.get_distribution(m=0, t=i)
                x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=100)
                opt_param_evol.append(10**x[np.argmax(pdf)]/df_bind.loc[df_bind["Variant"]==var,"_".join(param.split('_')[:-1])+"_f"].values[0])
            
            # Save distribution channging from unbinding rate to Kd
            x = 10**x/df_bind.loc[df_bind["Variant"]==var,"_".join(param.split('_')[:-1])+"_f"].values[0]
            param = "Kd_"+"_".join(param.split('_')[1:-1])
        else:
            # Change the binding rates dataset to include the fitted parameters
            param_dict[var__param] = x[np.argmax(pdf)]
            
            # Get the evolution of the optimal parameter values with each iteration of the posterior
            for i in range(0,history.max_t+1):
                df, w = history.get_distribution(m=0, t=i)
                x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=100)
                opt_param_evol.append(10**x[np.argmax(pdf)])
            
            # Save distribution channging from unbinding rate to Kd
            x = 10**x
            
        print("Optimal "+var+"_"+param+": " + str(x[np.argmax(pdf)]))    
        
        # Plot posterior
        plt.plot(x,pdf/np.sum(pdf), color="blue", linewidth=7.5, label=r"$p(\theta|D)$")

        # Finalize plot with prior and posterior distributions
        plt.ylabel("Probability", fontsize=25)
        plt.xlabel(var+"_"+param, fontsize=25)
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.legend(loc="best", fontsize=20)
        ax.set_yticks([])
        plt.xscale('log')
        plt.savefig(directory_plots +'Param_inference_distribution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.close()
        
        # Plot evolution of optimal parameter values
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        plt.plot(range(-1,history.max_t+1),opt_param_evol, color="blue", linewidth=7.5)
        plt.xlabel("Iteration", fontsize=25)
        plt.ylabel(var+"_"+param, fontsize=25)
        plt.yscale("log")
        if "Kd" in param:
            plt.ylim(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Lower_bound"].values[0]/df_bind.loc[df_bind["Variant"]==var,var__param.split("__")[1].replace("_b","_f")].values[0],df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Upper_bound"].values[0]/df_bind.loc[df_bind["Variant"]==var,var__param.split("__")[1].replace("_b","_f")].values[0])
            middle_tick = float('1e'+str(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]/df_bind.loc[df_bind["Variant"]==var,var__param.split("__")[1].replace("_b","_f")].values[0]).split('e')[1])
            plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
        else:
            plt.ylim(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Lower_bound"].values[0],df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Upper_bound"].values[0])
            middle_tick = float('1e'+str(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]).split('e')[1])
            plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.savefig(directory_plots +'Param_inference_evolution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.close()
        
    # Plot epsilon evolution
    x = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_xdata()
    y = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_ydata()
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.plot(x,y, color="blue", linewidth=7.5)
    plt.xlabel("Iteration", fontsize=25)
    plt.ylabel(r'$\epsilon$', fontsize=25)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    plt.savefig(directory_plots +'Param_inference_evolution_'+var+'__epsilon.pdf', transparent=True, bbox_inches="tight")
    plt.close()

Original WT_surf_cell: 1.3363678393265691e-08
Optimal WT_surf_cell: 1.7259854352188964e-09
Original WT_k_IL_RA_b: 1.010094423803225e-08
Optimal WT_Kd_IL_RA: 1.2166623094666402e-08
Original WT_k_IL_RB_b: 2.9200284002853823e-07
Optimal WT_Kd_IL_RB: 3.4363243229054864e-08
Original WT_k_IL_RG_b: 4.772742283309168e-05
Optimal WT_Kd_IL_RG: 6.164233697210332e-06


In [4]:
# Update binding rates to the infered values and save them
variants_list = ["WT","NKTR-358","MDNA11","ALKS4230","THOR-707"]
df_bind = update_df_bind_ABC(df_bind, param_dict, param_depen, param_unique_var, variants_list)
df_bind.to_csv("results/fit_ABC_SMC/IL2_data_param_ABC_SMC_WT.csv", index = False) # Save infered parameter values

# Plot and save parameter inference of WT + variants

In [6]:
directory_plots = "figures/ABC_SMC/"
conf95_2SD = 0.75
df_bind = pd.read_csv("data/binding/IL2_data_param.csv")
param_dict = {}
variants_list = ["WT","NKTR-358","MDNA11","ALKS4230","THOR-707"]
param_depen = {'k_IL_RB_b':['k_IL_RA_RB_b'], 'k_RA_RB_b':['k_IL_RA_RB_b'], 'k_IL_RB_RG_b':['k_IL_RA_RB_RG_b'], 'k_IL_RA_RB_RG_b':['k_IL_RB_RG_b']}
param_unique_var = {"WT": df_bind.columns[1:],"NKTR-358":['k_IL_RA_b','k_IL_RB_b','k_IL_RA_RB_b'],"WT_Nterm":['k_IL_RA_b','k_IL_RB_RG_b','k_IL_RA_RB_RG_b'],"MDNA11":['k_IL_RA_f','k_IL_RA_b','k_IL_RB_b'],"ALKS4230":['k_IL_RA_f','k_IL_RA_b'],"THOR-707":['k_IL_RA_f','k_IL_RA_b','k_IL_RB_f','k_IL_RB_b','k_IL_RA_RB_b']}

In [7]:
param_dict = {'vol_EC':'$V_{EC}$','surf_cell':'$S_{cell}$','vol_cell':'$V_{cell}$','k_RA_RB_b':'$Kd_{RA-RB}$','k_IL_RA_b':'$Kd_{IL-RA}$','k_IL_RB_b':'$Kd_{IL-RB}$','k_IL_RG_b':'$Kd_{IL-RG}$','k_IL_RA_RB_b':'$Kd_{IL-RA-RB}$','k_IL_RB_RG_b':'$Kd_{IL-RB-RG}$','k_IL_RA_RB_RG_b':'$Kd_{IL-RA-RB-RG}$','k_STAT5_b':'$Kd_{STAT5}$','k_PHOS':'$K_{PHOS}$','k_DEPHOS':'$Kd_{DEPHOS}$'}

In [8]:
for variant in variants_list:
    # Load parameter inference data
    history = pyabc.History("sqlite:///results/fit_ABC_SMC/results/pyABC_inference__"+variant+".db")
    df_fit_data = pd.read_csv("results/fit_ABC_SMC/IL2_data_fit.csv") # Parameters to fit
    df_fit_data["Var__Param"] = df_fit_data["Variant"]+"__"+df_fit_data["Parameter"]
    df_fit_data = df_fit_data[["Var__Param","Initial_val","Lower_bound","Upper_bound"]]
    df_fit_data = df_fit_data.loc[df_fit_data["Var__Param"].str.contains(variant)]
    
    # For each infered parameter in the variant
    for var__param in df_fit_data["Var__Param"]:
        var, param = var__param.split("__")
        param_plot = param_dict[param]

        # Plot prior and final posterior distributions
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        
        # Plot prior
        x = np.linspace(np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2,np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2,10000)
        pdf = pyabc.RV("norm", loc=np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]), scale=conf95_2SD/2).pdf(x)
        # If infered parameter is an unbinding rate in the plot we put the Kd
        if (param.split('_')[0]=='k') & (param.split('_')[-1]=='b'): 
            x = 10**x/df_bind.loc[df_bind["Variant"]==var,"_".join(param.split('_')[:-1])+"_f"].values[0]
        else:
            x = 10**x
        opt_param_evol = [x[np.argmax(pdf)]] # Save value where prior is maximum for temporal evolution    
            
        # Plot prior
        plt.plot(x,pdf/np.sum(pdf), color="darkred", linewidth=7.5, label=r"$p(\theta)$")
        
        print("Original "+var+"_"+param+": " + str(x[np.argmax(pdf)]))
        
        # Plot posterior
        df, w = history.get_distribution(m=0, t=history.max_t)
        x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
        # If infered parameter is an unbinding rate in the plot we put the Kd
        if (param.split('_')[0]=='k') & (param.split('_')[-1]=='b'):
            # Change the binding rates dataset to include the fitted parameters
            param_dict[var__param] = x[np.argmax(pdf)]
            
            # Get the evolution of the optimal parameter values with each iteration of the posterior
            for i in range(0,history.max_t+1):
                df, w = history.get_distribution(m=0, t=i)
                x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
                opt_param_evol.append(10**x[np.argmax(pdf)]/df_bind.loc[df_bind["Variant"]==var,"_".join(param.split('_')[:-1])+"_f"].values[0])
            
            # Save distribution channging from unbinding rate to Kd
            x = 10**x/df_bind.loc[df_bind["Variant"]==var,"_".join(param.split('_')[:-1])+"_f"].values[0]
            param = "Kd_"+"_".join(param.split('_')[1:-1])
        else:
            # Change the binding rates dataset to include the fitted parameters
            param_dict[var__param] = x[np.argmax(pdf)]
            
            # Get the evolution of the optimal parameter values with each iteration of the posterior
            for i in range(0,history.max_t+1):
                df, w = history.get_distribution(m=0, t=i)
                x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
                opt_param_evol.append(10**x[np.argmax(pdf)])
            
            # Save distribution channging from unbinding rate to Kd
            x = 10**x
            
        print("Optimal "+var+"_"+param+": " + str(x[np.argmax(pdf)]))    
        
        # Plot posterior
        plt.plot(x,pdf/np.sum(pdf), color="blue", linewidth=7.5, label=r"$p(\theta|D)$")

        # Finalize plot with prior and posterior distributions
        plt.ylabel("Probability", fontsize=25)
        plt.xlabel(param_plot, fontsize=25)
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.legend(loc="best", fontsize=20)
        ax.set_yticks([])
        plt.xscale('log')
        plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
        plt.savefig(directory_plots +'Param_inference_distribution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.close()
        
        # Plot evolution of optimal parameter values
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        plt.plot(range(-1,history.max_t+1),opt_param_evol, color="blue", linewidth=7.5)
        plt.xlabel("Iteration", fontsize=25)
        plt.ylabel(param_plot, fontsize=25)
        plt.yscale("log")
        if "Kd" in param:
            plt.ylim(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Lower_bound"].values[0]/df_bind.loc[df_bind["Variant"]==var,var__param.split("__")[1].replace("_b","_f")].values[0],df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Upper_bound"].values[0]/df_bind.loc[df_bind["Variant"]==var,var__param.split("__")[1].replace("_b","_f")].values[0])
            middle_tick = 10**round(np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]/df_bind.loc[df_bind["Variant"]==var,var__param.split("__")[1].replace("_b","_f")].values[0]))
            plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
        else:
            plt.ylim(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Lower_bound"].values[0],df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Upper_bound"].values[0])
            middle_tick = 10**round(np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]))
            plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.savefig(directory_plots +'Param_inference_evolution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.close()
        
    # Plot epsilon evolution
    x = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_xdata()
    y = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_ydata()
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.plot(x,y, color="blue", linewidth=7.5)
    plt.xlabel("Iteration", fontsize=25)
    plt.ylabel(r'$\epsilon$', fontsize=25)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    plt.savefig(directory_plots +'Param_inference_evolution_'+var+'__epsilon.pdf', transparent=True, bbox_inches="tight")
    plt.close()

Original WT_surf_cell: 1.3993553601551832e-08
Optimal WT_surf_cell: 1.7596972959355147e-09
Original WT_k_IL_RA_b: 1.0577035787722892e-08
Optimal WT_Kd_IL_RA: 1.204396859394035e-08
Original WT_k_IL_RB_b: 3.0576591814749415e-07
Optimal WT_Kd_IL_RB: 3.5099026104527814e-08
Original WT_k_IL_RG_b: 4.997697714839932e-05
Optimal WT_Kd_IL_RG: 6.001750469734062e-06
Original NKTR-358_k_IL_RA_b: 7.2966386636663065e-06
Optimal NKTR-358_Kd_IL_RA: 1.5949857360417574e-05
Original NKTR-358_k_IL_RB_b: 8.795947978118217e-05
Optimal NKTR-358_Kd_IL_RB: 1.4015315678259126e-05
Original MDNA11_k_IL_RB_b: 6.596960983588569e-09
Optimal MDNA11_Kd_IL_RB: 1.5692117424402642e-09
Original ALKS4230_k_IL_RB_b: 3.0576591814749415e-07
Optimal ALKS4230_Kd_IL_RB: 3.658447830830403e-08
Original THOR-707_k_IL_RA_b: 0.00015792724778894196
Optimal THOR-707_Kd_IL_RA: 2.530332181434888e-05
Original THOR-707_k_IL_RB_b: 2.0890376448030927e-06
Optimal THOR-707_Kd_IL_RB: 2.0943360875114382e-07


In [ ]:
# Get mean distance of literature vs infered parameter values
dist_list = []
for var__param in param_dict.keys():
    var, param = var__param.split("__")
    dist_list.append(abs(param_dict[var__param]-np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])))
print("Mean absolute distance: "+str(np.mean(dist_list)))

In [8]:
# Update binding rates to the infered values and save them
variants_list = ["WT","NKTR-358","MDNA11","ALKS4230","THOR-707"]
df_bind = update_df_bind_ABC(df_bind, param_dict, param_depen, param_unique_var, variants_list)
df_bind.to_csv("results/fit_ABC_SMC/IL2_data_param_ABC_SMC.csv", index = False) # Save infered parameter values

# Plot and save parameter inference of BEMPEG + variants

In [9]:
directory_plots = "figures/ABC_SMC/BEMPEG/"
conf95_2SD = 0.75
df_bind = pd.read_csv('data/binding/IL2_data_param_BEMPEG.csv')
param_dict = {}
variants_list = ["1-PEG-IL2","2-PEG-IL2"]
param_depen = {'k_IL_RB_b':['k_IL_RA_RB_b'], 'k_RA_RB_b':['k_IL_RA_RB_b'], 'k_IL_RB_RG_b':['k_IL_RA_RB_RG_b'], 'k_IL_RA_RB_RG_b':['k_IL_RB_RG_b']}
param_unique_var = {"WT": df_bind.columns[1:], "1-PEG-IL2":['k_IL_RA_b','k_IL_RB_b','k_IL_RA_RB_b'], "2-PEG-IL2":['k_IL_RA_b','k_IL_RB_b','k_IL_RA_RB_b']}

In [10]:
param_dict = {'vol_EC':'$V_{EC}$','surf_cell':'$S_{cell}$','vol_cell':'$V_{cell}$','k_RA_RB_b':'$Kd_{RA-RB}$','k_IL_RA_b':'$Kd_{IL-RA}$','k_IL_RB_b':'$Kd_{IL-RB}$','k_IL_RG_b':'$Kd_{IL-RG}$','k_IL_RA_RB_b':'$Kd_{IL-RA-RB}$','k_IL_RB_RG_b':'$Kd_{IL-RB-RG}$','k_IL_RA_RB_RG_b':'$Kd_{IL-RA-RB-RG}$','k_STAT5_b':'$Kd_{STAT5}$','k_PHOS':'$K_{PHOS}$','k_DEPHOS':'$Kd_{DEPHOS}$'}

In [11]:
for variant in variants_list:
    # Load parameter inference data
    history = pyabc.History("sqlite:///results/fit_ABC_SMC/results/pyABC_inference__"+variant+".db")
    df_fit_data = pd.read_csv('results/fit_ABC_SMC/BEMPEG_data_fit.csv')  # Parameters to fit
    df_fit_data = df_fit_data.loc[df_fit_data["Parameter"]!="k_IL_RA_RB_b"]
    df_fit_data["Var__Param"] = df_fit_data["Variant"]+"__"+df_fit_data["Parameter"]
    df_fit_data = df_fit_data[["Var__Param","Initial_val","Lower_bound","Upper_bound"]]
    df_fit_data = df_fit_data.loc[df_fit_data["Var__Param"].str.contains(variant)]
    
    # For each infered parameter in the variant
    for var__param in df_fit_data["Var__Param"]:
        var, param = var__param.split("__")
        param_plot = param_dict[param]

        # Plot prior and final posterior distributions
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        
        # Plot prior
        x = np.linspace(np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2,np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2,10000)
        pdf = pyabc.RV("norm", loc=np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]), scale=conf95_2SD/2).pdf(x)
        # If infered parameter is an unbinding rate in the plot we put the Kd
        if (param.split('_')[0]=='k') & (param.split('_')[-1]=='b'): 
            x = 10**x/df_bind.loc[df_bind["Variant"]==var,"_".join(param.split('_')[:-1])+"_f"].values[0]
        else:
            x = 10**x
        opt_param_evol = [x[np.argmax(pdf)]] # Save value where prior is maximum for temporal evolution    
            
        # Plot prior
        plt.plot(x,pdf/np.sum(pdf), color="darkred", linewidth=7.5, label=r"$p(\theta)$")
        
        print("Original "+var+"_"+param+": " + str(x[np.argmax(pdf)]))
        
        # Plot posterior
        df, w = history.get_distribution(m=0, t=history.max_t)
        x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
        # If infered parameter is an unbinding rate in the plot we put the Kd
        if (param.split('_')[0]=='k') & (param.split('_')[-1]=='b'):
            # Change the binding rates dataset to include the fitted parameters
            param_dict[var__param] = x[np.argmax(pdf)]
            
            # Get the evolution of the optimal parameter values with each iteration of the posterior
            for i in range(0,history.max_t+1):
                df, w = history.get_distribution(m=0, t=i)
                x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
                opt_param_evol.append(10**x[np.argmax(pdf)]/df_bind.loc[df_bind["Variant"]==var,"_".join(param.split('_')[:-1])+"_f"].values[0])
            
            # Save distribution channging from unbinding rate to Kd
            x = 10**x/df_bind.loc[df_bind["Variant"]==var,"_".join(param.split('_')[:-1])+"_f"].values[0]
            param = "Kd_"+"_".join(param.split('_')[1:-1])
        else:
            # Change the binding rates dataset to include the fitted parameters
            param_dict[var__param] = x[np.argmax(pdf)]
            
            # Get the evolution of the optimal parameter values with each iteration of the posterior
            for i in range(0,history.max_t+1):
                df, w = history.get_distribution(m=0, t=i)
                x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
                opt_param_evol.append(10**x[np.argmax(pdf)])
            
            # Save distribution channging from unbinding rate to Kd
            x = 10**x
            
        print("Optimal "+var+"_"+param+": " + str(x[np.argmax(pdf)]))    
        
        # Plot posterior
        plt.plot(x,pdf/np.sum(pdf), color="blue", linewidth=7.5, label=r"$p(\theta|D)$")

        # Finalize plot with prior and posterior distributions
        plt.ylabel("Probability", fontsize=25)
        plt.xlabel(param_plot, fontsize=25)
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.legend(loc="best", fontsize=20)
        ax.set_yticks([])
        plt.xscale('log')
        plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
        plt.savefig(directory_plots +'Param_inference_distribution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.close()
        
        # Plot evolution of optimal parameter values
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        plt.plot(range(-1,history.max_t+1),opt_param_evol, color="blue", linewidth=7.5)
        plt.xlabel("Iteration", fontsize=25)
        plt.ylabel(var+"_"+param, fontsize=25)
        plt.yscale("log")
        if "Kd" in param:
            plt.ylim(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Lower_bound"].values[0]/df_bind.loc[df_bind["Variant"]==var,var__param.split("__")[1].replace("_b","_f")].values[0],df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Upper_bound"].values[0]/df_bind.loc[df_bind["Variant"]==var,var__param.split("__")[1].replace("_b","_f")].values[0])
            middle_tick = 10**round(np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]/df_bind.loc[df_bind["Variant"]==var,var__param.split("__")[1].replace("_b","_f")].values[0]))
            plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
        else:
            plt.ylim(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Lower_bound"].values[0],df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Upper_bound"].values[0])
            middle_tick = 10**round(np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]))
            plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.savefig(directory_plots +'Param_inference_evolution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.close()
        
    # Plot epsilon evolution
    x = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_xdata()
    y = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_ydata()
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.plot(x,y, color="blue", linewidth=7.5)
    plt.xlabel("Iteration", fontsize=25)
    plt.ylabel(r'$\epsilon$', fontsize=25)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    plt.savefig(directory_plots +'Param_inference_evolution_'+var+'__epsilon.pdf', transparent=True, bbox_inches="tight")
    plt.close()

Original 1-PEG-IL2_k_IL_RA_b: 1.8958853275983423e-07
Optimal 1-PEG-IL2_Kd_IL_RA: 1.986616763166679e-08
Original 1-PEG-IL2_k_IL_RB_b: 1.7686887657483162e-06
Optimal 1-PEG-IL2_Kd_IL_RB: 1.8363398018672574e-07
Original 2-PEG-IL2_k_IL_RA_b: 4.864069433694479e-07
Optimal 2-PEG-IL2_Kd_IL_RA: 6.482047915549718e-08
Original 2-PEG-IL2_k_IL_RB_b: 3.949987591487949e-06
Optimal 2-PEG-IL2_Kd_IL_RB: 6.62091730473854e-07


In [53]:
# Update binding rates to the infered values and save them
variants_list = ["WT","1-PEG-IL2","2-PEG-IL2"]
df_bind = update_df_bind_ABC(df_bind, param_dict, param_depen, param_unique_var, variants_list)
df_bind.to_csv("results/fit_ABC_SMC/IL2_data_param_ABC_SMC_BEMPEG.csv", index = False) # Save infered parameter values

In [13]:
# Compare literature vs fitted for Kd_IL2RA and other rates
df_lit = pd.read_csv("data/binding/IL2_data_param.csv")
df_lit2 = pd.read_csv('data/binding/IL2_data_param_BEMPEG.csv')
df_fit = pd.read_csv("results/fit_ABC_SMC/IL2_data_param_ABC_SMC.csv")
df_fit2 = pd.read_csv("results/fit_ABC_SMC/IL2_data_param_ABC_SMC_BEMPEG.csv")
df_lit.replace(0, np.nan, inplace=True)
df_fit.replace(0, np.nan, inplace=True)
df_lit2.replace(0, np.nan, inplace=True)
df_fit2.replace(0, np.nan, inplace=True)

In [14]:
# Differnece between literature and fitted for RA binding
np.mean(((np.log10(df_lit.dropna()["k_IL_RA_b"])-np.log10(df_fit.dropna()["k_IL_RA_b"])).abs().to_list()+(np.log10(df_lit2.dropna()["k_IL_RA_b"])-np.log10(df_fit2.dropna()["k_IL_RA_b"])).abs().to_list()))

0.5077507750775098

In [15]:
# Differnece between literature and fitted for RB and RG binding
np.mean(((np.log10(df_lit.dropna()["k_IL_RB_b"])-np.log10(df_fit.dropna()["k_IL_RB_b"])).abs().to_list()+(np.log10(df_lit2.dropna()["k_IL_RB_b"])-np.log10(df_fit2.dropna()["k_IL_RB_b"])).abs().to_list()+(np.log10(df_lit.dropna()["k_IL_RG_b"])-np.log10(df_fit.dropna()["k_IL_RG_b"])).abs().to_list()))

0.8065695458434735

# Plot and save parameter inference of mutant search

In [2]:
directory_plots = "figures/ABC_SMC/mutant_search/"
conf95_2SD = 1.5
df_bind = pd.read_csv("results/fit_ABC_SMC/IL2_data_param_ABC_SMC.csv")
param_dict = {}
variants_list = ["WT","NKTR-358","MDNA11","ALKS4230","THOR-707"]
param_depen = {'k_IL_RB_b':['k_IL_RA_RB_b'], 'k_RA_RB_b':['k_IL_RA_RB_b'], 'k_IL_RB_RG_b':['k_IL_RA_RB_RG_b'], 'k_IL_RA_RB_RG_b':['k_IL_RB_RG_b']}
param_unique_var = {"WT": df_bind.columns[1:],"NKTR-358":['k_IL_RA_b','k_IL_RB_b','k_IL_RA_RB_b'],"WT_Nterm":['k_IL_RA_b','k_IL_RB_RG_b','k_IL_RA_RB_RG_b'],"MDNA11":['k_IL_RA_f','k_IL_RA_b','k_IL_RB_b'],"ALKS4230":['k_IL_RA_f','k_IL_RA_b'],"THOR-707":['k_IL_RA_f','k_IL_RA_b','k_IL_RB_f','k_IL_RB_b','k_IL_RA_RB_b']}

In [3]:
param_dict = {'vol_EC':'$V_{EC}$','surf_cell':'$S_{cell}$','vol_cell':'$V_{cell}$','k_RA_RB_b':'$Kd_{RA-RB}$','k_IL_RA_b':'$Kd_{IL-RA}$','k_IL_RB_b':'$Kd_{IL-RB}$','k_IL_RG_b':'$Kd_{IL-RG}$','k_IL_RA_RB_b':'$Kd_{IL-RA-RB}$','k_IL_RB_RG_b':'$Kd_{IL-RB-RG}$','k_IL_RA_RB_RG_b':'$Kd_{IL-RA-RB-RG}$','k_STAT5_b':'$Kd_{STAT5}$','k_PHOS':'$K_{PHOS}$','k_DEPHOS':'$Kd_{DEPHOS}$'}

In [4]:
# Load parameter inference data
history = pyabc.History("sqlite:///results/fit_ABC_SMC/results/pyABC_mutant_search_melanoma.db")
param_fit = ['k_IL_RA_b','k_IL_RB_b','k_IL_RG_b']
var = "Mut"

# For each infered parameter in the variant
for param in param_fit:
    param_plot = param
    # Plot prior and final posterior distributions
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    param_plot2 = param_dict[param]

    # Plot prior
    x = np.linspace(np.log10(df_bind.loc[df_bind["Variant"]=="WT",param].values[0])-2,np.log10(df_bind.loc[df_bind["Variant"]=="WT",param].values[0])+2,10000)
    pdf = pyabc.RV("uniform", np.log10(df_bind.loc[df_bind["Variant"]=="WT",param].values[0])-conf95_2SD, conf95_2SD*2).pdf(x)
    x = 10**x/df_bind.loc[df_bind["Variant"]=="WT","_".join(param.split('_')[:-1])+"_f"].values[0]
    
    
    opt_param_evol = [df_bind.loc[df_bind["Variant"]=="WT",param].values[0]/df_bind.loc[df_bind["Variant"]=="WT","_".join(param.split('_')[:-1])+"_f"].values[0]] # Save WT value for temporal evolution    

    # Plot prior
    plt.plot(x,pdf/np.sum(pdf), color="darkred", linewidth=7.5, label=r"$p(\theta)$")

    print("Original "+var+"_"+"Kd_"+"_".join(param.split('_')[1:-1])+": " + str(df_bind.loc[df_bind["Variant"]=="WT",param].values[0]/df_bind.loc[df_bind["Variant"]=="WT","_".join(param.split('_')[:-1])+"_f"].values[0]))

    # Plot posterior
    df, w = history.get_distribution(m=0, t=history.max_t)
    x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var+"__"+param,xmin = np.log10(df_bind.loc[df_bind["Variant"]=="WT",param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]=="WT",param].values[0])+2, numx=10000)
        
    # Change the binding rates dataset to include the fitted parameters
    param_dict["Mut__"+param] = x[np.argmax(pdf)]

    # Get the evolution of the optimal parameter values with each iteration of the posterior
    for i in range(0,history.max_t+1):
        df, w = history.get_distribution(m=0, t=i)
        x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var+"__"+param,xmin = np.log10(df_bind.loc[df_bind["Variant"]=="WT",param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]=="WT",param].values[0])+2, numx=10000)
        opt_param_evol.append(10**x[np.argmax(pdf)]/df_bind.loc[df_bind["Variant"]=="WT","_".join(param.split('_')[:-1])+"_f"].values[0])

    # Save distribution channging from unbinding rate to Kd
    x = 10**x/df_bind.loc[df_bind["Variant"]=="WT","_".join(param.split('_')[:-1])+"_f"].values[0]
    param = "Kd_"+"_".join(param.split('_')[1:-1])

    print("Optimal "+var+"_"+param+": " + str(x[np.argmax(pdf)]))    

    # Plot posterior
    plt.plot(x,pdf/np.sum(pdf), color="blue", linewidth=7.5, label=r"$p(\theta|D)$")

    # Finalize plot with prior and posterior distributions
    plt.ylabel("Probability", fontsize=25)
    plt.xlabel(param_plot2, fontsize=25)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    plt.legend(loc="best", fontsize=20)
    ax.set_yticks([])
    plt.xscale('log')
    plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    plt.savefig(directory_plots +'Param_inference_distribution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
    plt.close()

    # Plot evolution of optimal parameter values
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.plot(range(-1,history.max_t+1),opt_param_evol, color="blue", linewidth=7.5)
    plt.xlabel("Iteration", fontsize=25)
    plt.ylabel(param_plot2, fontsize=25)
    plt.yscale("log")
    #plt.ylim(df_bind.loc[df_bind["Variant"]=="WT",param_plot].values[0]/df_bind.loc[df_bind["Variant"]=="WT",param_plot.replace("_b","_f")].values[0]/10,df_bind.loc[df_bind["Variant"]=="WT",param_plot].values[0]/df_bind.loc[df_bind["Variant"]=="WT",param_plot.replace("_b","_f")].values[0]*10)
    middle_tick = 10**round(df_bind.loc[df_bind["Variant"]=="WT",param_plot].values[0]/df_bind.loc[df_bind["Variant"]=="WT",param_plot.replace("_b","_f")].values[0])
    #plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    plt.savefig(directory_plots +'Param_inference_evolution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
    plt.close()

# Plot epsilon evolution
x = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_xdata()
y = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_ydata()
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
plt.plot(x,y, color="blue", linewidth=7.5)
plt.xlabel("Iteration", fontsize=25)
plt.ylabel(r'$\epsilon$', fontsize=25)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.savefig(directory_plots +'Param_inference_evolution_'+var+'__epsilon.pdf', transparent=True, bbox_inches="tight")
plt.close()

Original Mut_Kd_IL_RA: 1.2043968593940348e-08
Optimal Mut_Kd_IL_RA: 6.482502779865353e-09
Original Mut_Kd_IL_RB: 3.509902610452778e-08
Optimal Mut_Kd_IL_RB: 1.3885284807799581e-09
Original Mut_Kd_IL_RG: 6.001750469734063e-06
Optimal Mut_Kd_IL_RG: 8.863288112330374e-07


In [21]:
# Save df_bind with optimized mutant
df_bind = df_bind.loc[[0]]
df_bind.loc[1] =df_bind.loc[0]
df_bind.loc[1,"Variant"] = "Mut"
param_unique_var = {"WT": df_bind.columns[1:],"Mut":['k_IL_RA_b','k_IL_RB_b','k_IL_RG_b']}
variants_list = ["WT","Mut"]
df_bind = update_df_bind_ABC(df_bind, param_dict, param_depen, param_unique_var, variants_list)
df_bind.to_csv("results/fit_ABC_SMC/IL2_data_param_ABC_SMC_mut_melanoma.csv", index = False) # Save infered parameter values

## Plot the distribution of EC50 and amplitude among prior and posterior distributions

In [2]:
import pandas as pd
from src.simulate_func import hill_func
from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)

In [3]:
df_prior = pd.read_csv("results/EC50_AMP_IL2s_prior.tsv.gz", compression='gzip', sep='\t')
df_prior["AMP"] = np.log10(df_prior["AMP"])
df_post = pd.read_csv("results/EC50_AMP_IL2s_post.tsv.gz", compression='gzip', sep='\t')
df_post["AMP"] = np.log10(df_post["AMP"])
df_EC50_AMP = pd.read_csv('data/signaling/IL2_data_hill.csv')
df_EC50_AMP["Amp_t"] = np.log10(df_EC50_AMP["Amp_t"])

In [4]:
# Separate experimental data into train and test
cells_test = ["T8.M", "T8.naive", "T4.M_IL2RAlow", "T4.naive_IL2RAlow", "mTregs", "nTregs","ACH-000061"]
df_test = df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"].isin(cells_test))&(df_EC50_AMP["Variant"]=="WT")]
df_train = df_EC50_AMP.loc[~((df_EC50_AMP["Cell_type"].isin(cells_test))&(df_EC50_AMP["Variant"]=="WT"))]

In [ ]:
sns.set_context("notebook", rc={"lines.linewidth": 3})
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
sns.kdeplot(x=df_prior["EC50"], y=df_prior["AMP"], fill=True, cmap="Reds", levels=np.linspace(0.001,1,5),alpha=0.75, zorder=1)
sns.kdeplot(x=df_prior["EC50"], y=df_prior["AMP"], levels=np.linspace(0.001,1,5),alpha=0.75, linestyles="dashed", c="Darkred", zorder=2)
sns.kdeplot(x=df_post["EC50"], y=df_post["AMP"], fill=True, cmap="Blues", levels=np.linspace(0.001,1,3),alpha=0.75, zorder=3, cut=5)
sns.kdeplot(x=df_post["EC50"], y=df_post["AMP"], levels=np.linspace(0.001,1,3),alpha=0.75, linestyles="dashdot", c="Darkblue", zorder=4, cut=5)
plt.scatter(df_train["EC50_t"], df_train["Amp_t"], marker="x", c="orange", s=155, alpha=0.75, zorder=5)
plt.scatter(df_test["EC50_t"], df_test["Amp_t"], marker="+", c="green", s=185, alpha=0.75, zorder=6)
legend_elements = [
    Line2D([0], [0], color='darkred', label='Prior distr.', linewidth=3, alpha=0.75, linestyle="dashed"),
    Line2D([0], [0], color='darkblue', label='Posterior distr.', linewidth=3, alpha=0.75, linestyle="dashdot"),
    Line2D([0], [0], marker="x", color='w', label='Training data', markeredgecolor='orange', markersize=17, alpha=0.75, markeredgewidth=3),
    Line2D([0], [0], marker="+", color='w', label='Test data', markeredgecolor='green', markersize=20, alpha=0.75, markeredgewidth=3)
]
plt.legend(handles=legend_elements, loc='lower left', fontsize=20)
plt.xlabel('EC50', fontsize=25)
plt.ylabel('Amplitude', fontsize=25)
plt.xticks([-12,-10,-8,-6,-4], ["$10^{-12}$","$10^{-10}$","$10^{-8}$","$10^{-6}$","$10^{-4}$"])
plt.yticks([1.5,2,2.5], [50,100,500])
plt.ylim(1.25, 2.75)
plt.xlim(-13,-3)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.savefig('figures/ABC_SMC/Prior_post_distribution_output.pdf', transparent=True, bbox_inches="tight")
plt.show()